# **1. Tratamento dos dados**

Este notebook mantém os jogadores nascidos no Brasil, padroniza os campos de local de nascimento, associa cada cidade a um município brasileiro e exporta as tabelas de análise e os arquivos JavaScript usados pela visualização. Jogadores nascidos no exterior ficam fora da tabela tratada, mas são contados na auditoria.

| camada | arquivos principais | papel |
|---|---|---|
| entrada | `data/processed/brasileirao_2026_player_birthplaces.csv`, `data/raw/transfermarkt-api/` | jogadores e locais de nascimento coletados |
| referência | `data/raw/reference/` | coordenadas dos municípios, estados e limites estaduais |
| saída | `data/processed/brasileirao_2026_brazilian_player_birthplaces.csv`, `brasileirao_2026_birthplace_cities.csv` | uma linha por jogador e uma por município |
| auditoria | `data/processed/brasileirao_2026_birthplace_treatment_audit.csv` | filtros e cobertura de municípios |
| visualização | `html/data/birthplaces.js`, `html/data/brazil-states.js` | dados lidos pelo `index.html` |

A sequência é: carregar referências → selecionar nascidos no Brasil → associar municípios → montar as tabelas → exportar.

In [1]:
from __future__ import annotations

import gzip
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import display

### **1.1. Parâmetros e entradas**

`SEASON` define a temporada tratada e `PLAYERS_PATH` aponta para a tabela produzida pelo notebook 00. `REFERENCE_URLS` lista as três fontes de referência, mantidas em cache em `data/raw/reference/`.

In [2]:
# A raiz do projeto é detectada automaticamente, dentro ou fora de `notebooks/`.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REFERENCE = RAW / "reference"
HTML_DATA = ROOT / "html" / "data"
for directory in (REFERENCE, PROCESSED, HTML_DATA):
    directory.mkdir(parents=True, exist_ok=True)

# Temporada tratada e arquivo de entrada produzido pelo notebook 00.
SEASON = 2026
PLAYERS_PATH = PROCESSED / f"brasileirao_{SEASON}_player_birthplaces.csv"
PROFILE_CACHE = RAW / "transfermarkt-api"

# Dados de referência: coordenadas dos municípios, estados e limites estaduais.
REFERENCE_URLS = {
    "brazilian_municipalities.csv": "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv",
    "brazilian_states.csv": "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/estados.csv",
    "brazil_states.geojson": "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR?formato=application/vnd.geo+json&qualidade=minima&intrarregiao=UF",
}

## **2. Dados geográficos de referência**

As coordenadas dos municípios vêm da tabela `municipios-brasileiros`; os limites estaduais, da API de malhas do IBGE. Arquivos já existentes são reaproveitados, então novas execuções são reproduzíveis e não dependem de rede.

In [3]:
def download_if_missing(filename: str, url: str) -> Path:
    """Baixa o arquivo de referência apenas se ele ainda não estiver em `data/raw/reference/`."""
    path = REFERENCE / filename
    if path.exists() and path.stat().st_size > 100:
        return path
    response = requests.get(
        url,
        headers={"User-Agent": "academic-research/brazilian-league-player-birthplaces"},
        timeout=90,
    )
    response.raise_for_status()
    path.write_bytes(response.content)
    return path


reference_paths = {
    name: download_if_missing(name, url)
    for name, url in REFERENCE_URLS.items()
}
print({name: path.stat().st_size for name, path in reference_paths.items()})

{'brazilian_municipalities.csv': 390201, 'brazilian_states.csv': 1077, 'brazil_states.geojson': 98502}


## **3. Seleção dos jogadores nascidos no Brasil**

O país de nascimento, e não a nacionalidade, define o universo do estudo. O campo adicional de local de nascimento do Transfermarkt é recuperado do cache bruto dos perfis, pois muitas vezes informa o estado e melhora a associação com o município.

In [4]:
def clean_text(value: Any) -> str | None:
    """Normaliza para Unicode NFC e remove espaços repetidos; vazio vira `None`."""
    if value is None or pd.isna(value):
        return None
    cleaned = " ".join(unicodedata.normalize("NFC", str(value)).strip().split())
    return cleaned or None


def normalize_key(value: Any) -> str | None:
    """Chave de comparação: sem acentos, em minúsculas e só com letras e números."""
    cleaned = clean_text(value)
    if cleaned is None:
        return None
    folded = "".join(
        character
        for character in unicodedata.normalize("NFKD", cleaned.casefold())
        if not unicodedata.combining(character)
    )
    return re.sub(r"[^a-z0-9]+", " ", folded).strip() or None


# O estado informado pelo Transfermarkt fica no cache dos perfis e ajuda a identificar o município.
players = pd.read_csv(PLAYERS_PATH, dtype={"player_id": "string", "club_ids": "string"})
state_rows: list[dict[str, Any]] = []
for path in sorted(PROFILE_CACHE.glob(f"players_{SEASON}_batch_*.json.gz")):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        profiles = json.load(handle)["payload"]["data"]
    for profile in profiles:
        birth = profile.get("birthPlaceDetails") or {}
        state_rows.append({
            "player_id": str(profile["id"]),
            "birth_state_raw": clean_text(birth.get("placeOfBirthAdditionalInfo")),
        })

states_from_profiles = pd.DataFrame(state_rows).drop_duplicates("player_id")
players = players.merge(states_from_profiles, on="player_id", how="left", validate="one_to_one")
for column in ["player_name", "full_name", "birth_city", "birth_state_raw", "clubs", "position_group", "position"]:
    players[column] = players[column].map(clean_text)

# O universo do estudo é definido pelo país de nascimento, não pela nacionalidade.
foreigners_removed = int(players.birth_country_fifa_code.ne("BRA").sum())
brazilian = players.loc[players.birth_country_fifa_code.eq("BRA")].copy()
brazilian["birth_city_raw"] = brazilian["birth_city"]
brazilian["city_key"] = brazilian.birth_city.map(normalize_key)
brazilian["state_key"] = brazilian.birth_state_raw.map(normalize_key).replace({
    "distrito federal do brasil": "distrito federal",
})
print(f"{len(players):,} jogadores coletados | {len(brazilian):,} nascidos no Brasil | {foreigners_removed:,} nascidos no exterior removidos")

668 jogadores coletados | 510 nascidos no Brasil | 158 nascidos no exterior removidos


## **4. Associação com os municípios**

A associação prioriza o par cidade–estado exato e depois nomes de cidade únicos no país. Uma tabela curta e explícita de correções trata erros de grafia, distritos informados como cidades e dois nomes de município ambíguos. Cada regra fica registrada em `geocoding_rule`, e nenhuma coordenada é inventada para cidades de nascimento ausentes.

| `geocoding_rule` | critério |
|---|---|
| `correcao_manual` | correção explícita de grafia, distrito ou ambiguidade |
| `cidade_estado_exato` | cidade e estado coincidem com a referência |
| `cidade_unica_exata` | sem estado informado; o nome existe em um só município |
| `cidade_unica_estado_divergente` | nome único, mas o estado da fonte diverge da referência |
| `cidade_ausente` | jogador sem cidade de nascimento; permanece sem mapa |

In [5]:
# Tabela de municípios com UF e região.
municipalities = pd.read_csv(reference_paths["brazilian_municipalities.csv"])
state_reference = pd.read_csv(reference_paths["brazilian_states.csv"], encoding="utf-8-sig")
municipalities = municipalities.merge(
    state_reference[["codigo_uf", "uf", "nome", "regiao"]],
    on="codigo_uf",
    how="left",
    validate="many_to_one",
    suffixes=("", "_estado"),
)
municipalities["city_key"] = municipalities.nome.map(normalize_key)
municipalities["state_key"] = municipalities.nome_estado.map(normalize_key)
# Índices de busca: par cidade–estado e cidades com nome único no país.
municipalities_by_pair = {
    (row.city_key, row.state_key): row
    for row in municipalities.itertuples(index=False)
}
city_name_counts = municipalities.groupby("city_key").size()
unique_municipalities = {
    row.city_key: row
    for row in municipalities.itertuples(index=False)
    if city_name_counts.loc[row.city_key] == 1
}

# Correção da fonte → município padronizado. As regras continuam visíveis na saída.
MANUAL_GEOCODING = {
    ("santa barbara", "goias"): ("santa barbara de goias", "goias", "nome do município completado"),
    ("santo antonio do salto da onca", "rio grande do norte"): ("santo antonio", "rio grande do norte", "nome histórico da localidade"),
    ("jotaesse", "parana"): ("tupassi", "parana", "distrito atribuído ao município"),
    ("campo dos goytacazes", "rio de janeiro"): ("campos dos goytacazes", "rio de janeiro", "correção de grafia"),
    ("itaperica da serra", "sao paulo"): ("itapecerica da serra", "sao paulo", "correção de grafia"),
    ("sao mateus", "sao paulo"): ("sao paulo", "sao paulo", "distrito atribuído ao município"),
    ("presidente dutra", None): ("presidente dutra", "maranhao", "cidade ambígua desambiguada"),
    ("alvorada", None): ("alvorada", "rio grande do sul", "cidade ambígua desambiguada"),
}

# Ordem de prioridade: correção manual → par cidade–estado → cidade de nome único.
geocoded_rows: list[dict[str, Any]] = []
for player in brazilian.to_dict(orient="records"):
    city_key = player.get("city_key")
    state_key = player.get("state_key")
    city_key = None if pd.isna(city_key) else city_key
    state_key = None if pd.isna(state_key) else state_key
    municipality = None  # nenhuma coordenada é inventada para cidades ausentes
    rule = "sem_correspondencia"
    note = None

    override = MANUAL_GEOCODING.get((city_key, state_key)) or MANUAL_GEOCODING.get((city_key, None))
    if city_key is None:
        rule = "cidade_ausente"
    elif override:
        target_city, target_state, note = override
        municipality = municipalities_by_pair[(target_city, target_state)]
        rule = "correcao_manual"
    elif state_key is not None and (city_key, state_key) in municipalities_by_pair:
        municipality = municipalities_by_pair[(city_key, state_key)]
        rule = "cidade_estado_exato"
    elif city_key in unique_municipalities:
        municipality = unique_municipalities[city_key]
        rule = "cidade_unica_exata" if state_key is None else "cidade_unica_estado_divergente"
        if state_key is not None:
            note = "município único utilizado; o estado da fonte diverge da referência"

    result = dict(player)
    result["geocoding_rule"] = rule
    result["geocoding_note"] = note
    result["map_status"] = "nao_mapeado"
    result.update({
        "municipality_id": None,
        "birth_city": clean_text(player.get("birth_city")),
        "birth_state": clean_text(player.get("birth_state_raw")),
        "birth_state_code": None,
        "birth_region": None,
        "latitude": None,
        "longitude": None,
    })
    if municipality is not None:
        result.update({
            "municipality_id": int(municipality.codigo_ibge),
            "birth_city": municipality.nome,
            "birth_state": municipality.nome_estado,
            "birth_state_code": municipality.uf,
            "birth_region": municipality.regiao,
            "latitude": float(municipality.latitude),
            "longitude": float(municipality.longitude),
            "map_status": "mapeado",
        })
    geocoded_rows.append(result)

# Verificações da etapa; se alguma falhar, as regras de correspondência precisam de revisão.
brazilian_clean = pd.DataFrame(geocoded_rows)
brazilian_clean["municipality_id"] = brazilian_clean["municipality_id"].astype("Int64")
assert brazilian_clean.player_id.is_unique
assert len(brazilian_clean) == 510, f"Quantidade inesperada de jogadores nascidos no Brasil: {len(brazilian_clean)}"
assert brazilian_clean.map_status.eq("mapeado").sum() == 503, "A cobertura de municípios mudou; revise as regras de correspondência"
assert brazilian_clean.loc[brazilian_clean.map_status.ne("mapeado"), "geocoding_rule"].eq("cidade_ausente").all()
display(brazilian_clean.geocoding_rule.value_counts().rename("jogadores").to_frame())

,jogadores
geocoding_rule,
cidade_estado_exato,384
cidade_unica_exata,110
correcao_manual,8
cidade_ausente,7
cidade_unica_estado_divergente,1


## **5. Tabelas da visualização**

A tabela principal mantém uma linha por jogador nascido no Brasil. Uma segunda tabela agrega os jogadores mapeados por município, para dimensionar símbolos e montar o ranking. As contagens são de jogadores, não de linhas jogador–clube.

In [6]:
# Colunas finais da tabela de jogadores nascidos no Brasil.
clean_columns = [
    "season", "competition_id", "player_id", "player_name", "full_name",
    "date_of_birth", "birth_city_raw", "birth_state_raw", "birth_city",
    "birth_state", "birth_state_code", "birth_region", "municipality_id",
    "latitude", "longitude", "clubs", "club_ids", "position_group",
    "position", "profile_url", "map_status", "geocoding_rule", "geocoding_note",
]
brazilian_clean = (
    brazilian_clean[clean_columns]
    .sort_values(["map_status", "birth_state_code", "birth_city", "player_name"], na_position="last")
    .reset_index(drop=True)
)
mapped = brazilian_clean.loc[brazilian_clean.map_status.eq("mapeado")].copy()
# Agregação por município: contagem de jogadores e lista de nomes.
cities = (
    mapped.groupby(
        ["municipality_id", "birth_city", "birth_state", "birth_state_code", "birth_region", "latitude", "longitude"],
        as_index=False,
    )
    .agg(
        player_count=("player_id", "nunique"),
        players=("player_name", lambda values: " | ".join(sorted(values))),
    )
    .sort_values(["player_count", "birth_city"], ascending=[False, True])
    .reset_index(drop=True)
)
cities.insert(0, "city_rank", range(1, len(cities) + 1))
assert int(cities.player_count.sum()) == len(mapped)
display(cities.head(15))

,city_rank,municipality_id,birth_city,birth_state,birth_state_code,birth_region,latitude,longitude,player_count,players
0,1,3550308,São Paulo,São Paulo,SP,Sudeste,-23.53290,-46.6395,69,Ademir | Alan Patrick | Allex | André | Arthur...
1,2,3304557,Rio de Janeiro,Rio de Janeiro,RJ,Sudeste,-22.91290,-43.2003,40,Allan | Bruno Gomes | Bruno Tubarão | Carvalhe...
2,3,3106200,Belo Horizonte,Minas Gerais,MG,Sudeste,-19.91020,-43.9266,14,Bernard | Breno Lopes | Bruno Henrique | Edu R...
3,4,2927408,Salvador,Bahia,BA,Nordeste,-12.97180,-38.5011,14,Anderson Pato | Danilo | Eduardo Melo | Eric R...
4,5,2304400,Fortaleza,Ceará,CE,Nordeste,-3.71664,-38.5423,13,Artur | Da Silva | Dyogo Alves | Erick Pulga |...
5,6,4314902,Porto Alegre,Rio Grande do Sul,RS,Sul,-30.03180,-51.2065,8,Edenilson | Eduardo Sasha | Jáderson | Léo Ort...
6,7,4106902,Curitiba,Paraná,PR,Sul,-25.41950,-49.2646,7,Doma | Dudu | Gustavo Silva | Léo Andrade | Lé...
7,8,3543402,Ribeirão Preto,São Paulo,SP,Sudeste,-21.16990,-47.8099,6,Alex | Bruno Leonardo | Eduardo | Léo Jardim |...
8,9,3509502,Campinas,São Paulo,SP,Sudeste,-22.90530,-47.0659,5,Elias Manoel | Gabriel | Gustavo Scarpa | Rich...
9,10,5208707,Goiânia,Goiás,GO,Centro-Oeste,-16.68640,-49.2643,5,Dudu | Gustavo Neves | JP | Pedro Cobra | Zé V...


## **6. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/processed/brasileirao_2026_brazilian_player_birthplaces.csv` | nascidos no Brasil, com município, UF, região, coordenadas e regra aplicada |
| `data/processed/brasileirao_2026_birthplace_cities.csv` | um município por linha, com a contagem de jogadores |
| `data/processed/brasileirao_2026_birthplace_treatment_audit.csv` | contagens e cobertura da etapa |
| `html/data/birthplaces.js` | jogadores mapeados, em `window.BIRTHPLACE_DATA` |
| `html/data/brazil-states.js` | limites estaduais, em `window.BRAZIL_STATES` |

Os arquivos JavaScript expõem os mesmos registros validados como variáveis globais, de modo que o `index.html` funciona em um servidor estático simples, sem depender do ambiente Python.

In [7]:
# Auditoria do tratamento.
generated_at = datetime.now(timezone.utc).isoformat()
audit = pd.DataFrame([
    {"metric": "collected_players", "value": len(players)},
    {"metric": "foreign_born_removed", "value": foreigners_removed},
    {"metric": "brazilian_born_players", "value": len(brazilian_clean)},
    {"metric": "mapped_players", "value": int(brazilian_clean.map_status.eq("mapeado").sum())},
    {"metric": "unmapped_missing_birth_city", "value": int(brazilian_clean.map_status.ne("mapeado").sum())},
    {"metric": "mapped_municipalities", "value": int(len(cities))},
    {"metric": "municipality_coverage_pct", "value": round(brazilian_clean.map_status.eq("mapeado").mean() * 100, 2)},
])

brazilian_clean.to_csv(PROCESSED / f"brasileirao_{SEASON}_brazilian_player_birthplaces.csv", index=False)
cities.to_csv(PROCESSED / f"brasileirao_{SEASON}_birthplace_cities.csv", index=False)
audit.to_csv(PROCESSED / f"brasileirao_{SEASON}_birthplace_treatment_audit.csv", index=False)

# Dados da visualização: só jogadores mapeados, expostos como variáveis globais.
site_players = mapped.rename(columns={"clubs": "club"})[[
    "player_id", "player_name", "full_name", "date_of_birth",
    "birth_city", "birth_state", "birth_state_code", "birth_region",
    "municipality_id", "latitude", "longitude", "club",
    "position_group", "position", "profile_url",
]].to_dict(orient="records")
site_payload = {
    "metadata": {
        "season": SEASON,
        "competition": "Campeonato Brasileiro Série A",
        "collected_players": int(len(players)),
        "brazilian_born_players": int(len(brazilian_clean)),
        "foreign_born_removed": foreigners_removed,
        "mapped_players": int(len(mapped)),
        "unmapped_players": int(len(brazilian_clean) - len(mapped)),
        "mapped_municipalities": int(len(cities)),
        "generated_at_utc": generated_at,
        "source": "Transfermarkt; coordenadas dos municípios do repositório municipios-brasileiros; limites estaduais do IBGE",
    },
    "players": site_players,
}
(HTML_DATA / "birthplaces.js").write_text(
    "window.BIRTHPLACE_DATA = " + json.dumps(site_payload, ensure_ascii=False, separators=(",", ":")) + ";\n",
    encoding="utf-8",
)

# Limites estaduais com UF, nome e região, no sentido de anel esperado pelo D3.
with open(reference_paths["brazil_states.geojson"], encoding="utf-8") as handle:
    states_geojson = json.load(handle)
state_by_code = {str(row.codigo_uf): row for row in state_reference.itertuples(index=False)}
for feature in states_geojson["features"]:
    state = state_by_code[str(feature["properties"]["codarea"])]
    feature["properties"].update({"uf": state.uf, "name": state.nome, "region": state.regiao})
    # Os anéis do IBGE são anti-horários (RFC 7946); os polígonos esféricos do D3
    # esperam o sentido oposto para feições menores que um hemisfério.
    geometry = feature["geometry"]
    if geometry["type"] == "Polygon":
        geometry["coordinates"] = [list(reversed(ring)) for ring in geometry["coordinates"]]
    elif geometry["type"] == "MultiPolygon":
        geometry["coordinates"] = [
            [list(reversed(ring)) for ring in polygon]
            for polygon in geometry["coordinates"]
        ]
(HTML_DATA / "brazil-states.js").write_text(
    "window.BRAZIL_STATES = " + json.dumps(states_geojson, ensure_ascii=False, separators=(",", ":")) + ";\n",
    encoding="utf-8",
)

display(audit)
print(f"CSVs processados: {PROCESSED}")
print(f"Dados da visualização: {HTML_DATA}")

,metric,value
0,collected_players,668.00
1,foreign_born_removed,158.00
2,brazilian_born_players,510.00
3,mapped_players,503.00
4,unmapped_missing_birth_city,7.00
5,mapped_municipalities,244.00
6,municipality_coverage_pct,98.63


CSVs processados: /Users/gabrielcarbinatto/Documents/GitHub/brazilian-league-2026-player-birthplaces/data/processed
Dados da visualização: /Users/gabrielcarbinatto/Documents/GitHub/brazilian-league-2026-player-birthplaces/html/data
